# Customer Retention & Revenue Optimization Analytics

## Importing Libraries and loading Datasets

In [84]:
import pandas as pd
import numpy as np

In [85]:
orders = pd.read_csv(r"C:\Users\ADMIN\Desktop\olist-ecommerce-analytics\Data\Raw\olist_orders_dataset.csv")
order_items = pd.read_csv(r"C:\Users\ADMIN\Desktop\olist-ecommerce-analytics\Data\Raw\olist_order_items_dataset.csv")

### Changing DataTypes and Formats

In [86]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

In [87]:
orders['is_delivered'] = orders['order_delivered_customer_date'].notnull()

In [88]:
orders['delivery_time'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

In [89]:
orders['approval_time'] = (
    orders['order_approved_at'] - orders['order_purchase_timestamp']
).dt.days

In [90]:
orders['shipping_time'] = (
    orders['order_delivered_carrier_date'] - orders['order_approved_at']
).dt.days

In [91]:
orders.drop_duplicates(inplace=True)

In [135]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
Index: 94237 entries, 0 to 99440
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       94237 non-null  object        
 1   customer_id                    94237 non-null  object        
 2   order_status                   94237 non-null  object        
 3   order_purchase_timestamp       94237 non-null  datetime64[ns]
 4   order_approved_at              94237 non-null  datetime64[ns]
 5   order_delivered_carrier_date   94237 non-null  datetime64[ns]
 6   order_delivered_customer_date  94237 non-null  datetime64[ns]
 7   order_estimated_delivery_date  94237 non-null  datetime64[ns]
 8   is_delivered                   94237 non-null  bool          
 9   delivery_time                  94237 non-null  float64       
 10  approval_time                  94237 non-null  float64       
 11  shipping_time       

In [92]:
orders.describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time,approval_time,shipping_time
count,99441,99281,97658,96476,99441,96476.000000,99281.000000,97644.000000
mean,2017-12-31 08:43:12.776581120,2017-12-31 18:35:24.098800128,2018-01-04 21:49:48.138278656,2018-01-14 12:09:19.035542272,2018-01-24 03:08:37.730111232,12.094086,0.269800,2.301749
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30 00:00:00,0.000000,0.000000,-172.000000
25%,2017-09-12 14:46:19,2017-09-12 23:24:16,2017-09-15 22:28:50.249999872,2017-09-25 22:07:22.249999872,2017-10-03 00:00:00,6.000000,0.000000,0.000000
50%,2018-01-18 23:04:36,2018-01-19 11:36:13,2018-01-24 16:10:58,2018-02-02 19:28:10.500000,2018-02-15 00:00:00,10.000000,0.000000,1.000000
75%,2018-05-04 15:42:16,2018-05-04 20:35:10,2018-05-08 13:37:45,2018-05-15 22:48:52.249999872,2018-05-25 00:00:00,15.000000,0.000000,3.000000
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12 00:00:00,209.000000,187.000000,125.000000
std,NaN,NaN,NaN,NaN,NaN,9.551746,0.986202,3.560283


### Handling Missing Values

In [93]:
orders[orders['shipping_time'] < 0]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,delivery_time,approval_time,shipping_time
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04,True,13.0,5.0,-2.0
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15,True,2.0,2.0,-1.0
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31,True,4.0,5.0,-3.0
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31,True,1.0,0.0,-1.0
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06,True,5.0,5.0,-4.0
...,...,...,...,...,...,...,...,...,...,...,...,...
99091,240ead1a7284667e0ec71d01f80e4d5e,fcdd7556401aaa1c980f8b67a69f95dc,delivered,2018-07-02 16:30:02,2018-07-05 16:17:59,2018-07-05 14:11:00,2018-07-10 23:21:47,2018-07-24,True,8.0,2.0,-1.0
99230,78008d03bd8ef7fcf1568728b316553c,043e3254e68daf7256bda1c9c03c2286,delivered,2018-07-03 13:11:13,2018-07-05 16:32:52,2018-07-03 12:57:00,2018-07-10 17:47:39,2018-07-23,True,7.0,2.0,-3.0
99266,76a948cd55bf22799753720d4545dd2d,3f20a07b28aa252d0502fe7f7eb030a9,delivered,2018-01-30 02:41:30,2018-02-04 23:31:46,2018-01-31 18:11:58,2018-03-18 20:08:50,2018-03-02,True,47.0,5.0,-5.0
99377,a6bd1f93b7ff72cc348ca07f38ec4bee,6d63fa86bd2f62908ad328325799152f,delivered,2018-04-20 17:28:40,2018-04-24 19:26:10,2018-04-23 17:18:40,2018-04-28 17:38:42,2018-05-15,True,8.0,4.0,-2.0


In [94]:
orders[orders['shipping_time'] < 0].shape

(1359, 12)

In [95]:
orders = orders[orders['shipping_time'] >= 0]

In [96]:
orders[orders['delivery_time'] < 0]
orders[orders['approval_time'] < 0]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,delivery_time,approval_time,shipping_time


In [97]:
orders['delivery_time'].sort_values(ascending=False).head(10)

19590    209.0
55619    208.0
61610    195.0
70307    194.0
89130    194.0
38509    194.0
11399    191.0
81401    189.0
54480    188.0
68769    187.0
Name: delivery_time, dtype: float64

In [98]:
orders['delivery_time'].describe(percentiles=[0.90, 0.95, 0.99])

count    95111.000000
mean        12.151959
std          9.577141
min          0.000000
50%         10.000000
90%         23.000000
95%         29.000000
99%         46.000000
max        209.000000
Name: delivery_time, dtype: float64

In [99]:
upper_limit = orders['delivery_time'].quantile(0.99)

In [100]:
orders = orders[orders['delivery_time'] <= upper_limit]

In [101]:
orders[orders['delivery_time'] > 100]['order_status'].value_counts()

Series([], Name: count, dtype: int64)

In [102]:
orders['delivery_time'].max()

46.0

In [103]:
customers = pd.read_csv(r"C:\Users\ADMIN\Desktop\olist-ecommerce-analytics\Data\Raw\olist_customers_dataset.csv")
customers.info()
customers.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [104]:
customers['customer_city'] = customers['customer_city'].str.lower().str.strip()
customers['customer_state'] = customers['customer_state'].str.lower().str.strip()

In [105]:
customers.duplicated().sum()

np.int64(0)

In [106]:
customers['customer_unique_id'].nunique()


96096

In [107]:
products = pd.read_csv(r"C:\Users\ADMIN\Desktop\olist-ecommerce-analytics\Data\Raw\olist_products_dataset.csv")
products.info()
products.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [108]:
products = products.dropna(subset=[
    'product_category_name',
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty'
]).copy()

In [109]:
for col in cols:
    products[col] = products[col].fillna(products[col].median())

In [110]:
customers.head(10)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,sp
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,sp
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,sp
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,sp
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,sp
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,sc
6,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,sp
7,5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,mg
8,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,pr
9,4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,mg


### Merging Tables

In [111]:
orders_customers = orders.merge(
    customers,
    on='customer_id',
    how='left'
)

In [112]:
orders_full = orders_customers.merge(
    order_items,
    on='order_id',
    how='left'
)

In [113]:
master_df = orders_full.merge(
    products,
    on='product_id',
    how='left'
)

In [114]:
master_df.shape
master_df.columns
master_df.head()
master_df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   0
order_delivered_carrier_date        0
order_delivered_customer_date       0
order_estimated_delivery_date       0
is_delivered                        0
delivery_time                       0
approval_time                       0
shipping_time                       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
product_category_name            1518
product_name_lenght              1518
product_description_lenght       1518
product_photos_qty               1518
product_weig

### Handling Missing Values in Master Table

In [115]:
master_df = master_df.dropna(subset=['product_id']).copy()

In [116]:
master_df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   0
order_delivered_carrier_date        0
order_delivered_customer_date       0
order_estimated_delivery_date       0
is_delivered                        0
delivery_time                       0
approval_time                       0
shipping_time                       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
product_category_name            1518
product_name_lenght              1518
product_description_lenght       1518
product_photos_qty               1518
product_weig

In [117]:
master_df = master_df.dropna(subset=['product_category_name']).copy()

In [118]:
master_df.isnull().sum()

order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
is_delivered                     0
delivery_time                    0
approval_time                    0
shipping_time                    0
customer_unique_id               0
customer_zip_code_prefix         0
customer_city                    0
customer_state                   0
order_item_id                    0
product_id                       0
seller_id                        0
shipping_limit_date              0
price                            0
freight_value                    0
product_category_name            0
product_name_lenght              0
product_description_lenght       0
product_photos_qty               0
product_weight_g                 0
product_length_cm                0
product_height_cm   

In [119]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date'
]

for col in date_cols:
    master_df[col] = pd.to_datetime(master_df[col])

In [120]:
master_df['delivery_time'] = (
    master_df['order_delivered_customer_date'] - 
    master_df['order_purchase_timestamp']
).dt.days

In [121]:
master_df['approval_time'] = (
    master_df['order_approved_at'] - 
    master_df['order_purchase_timestamp']
).dt.days

In [122]:
master_df['shipping_time'] = (
    master_df['order_delivered_carrier_date'] - 
    master_df['order_approved_at']
).dt.days

In [123]:
master_df['total_cost'] = master_df['price'] + master_df['freight_value']

In [124]:
master_df[['delivery_time','approval_time','shipping_time']].describe()

,delivery_time,approval_time,shipping_time
count,106160.000000,106160.000000,106160.000000
mean,11.602515,0.247297,2.336888
std,7.775435,0.686568,3.228252
min,0.000000,0.000000,0.000000
25%,6.000000,0.000000,0.000000
50%,10.000000,0.000000,1.000000
75%,15.000000,0.000000,3.000000
max,46.000000,30.000000,43.000000


In [125]:
master_df = master_df[master_df['shipping_time'] >= 0]

In [126]:
master_df[['delivery_time','approval_time','shipping_time']].describe()

,delivery_time,approval_time,shipping_time
count,106160.000000,106160.000000,106160.000000
mean,11.602515,0.247297,2.336888
std,7.775435,0.686568,3.228252
min,0.000000,0.000000,0.000000
25%,6.000000,0.000000,0.000000
50%,10.000000,0.000000,1.000000
75%,15.000000,0.000000,3.000000
max,46.000000,30.000000,43.000000


In [127]:
master_df.loc[:, 'shipping_time'] = master_df['shipping_time'].clip(upper=30)

master_df.loc[:, 'total_cost'] = master_df['price'] + master_df['freight_value']

In [128]:
master_df.groupby('customer_state')['total_cost'] \
    .sum() \
    .sort_values(ascending=False) \
    .head(10)

customer_state
sp    5597950.99
rj    1955825.20
mg    1763516.70
rs     830189.32
pr     757207.51
sc     576694.37
ba     561599.88
df     333491.89
go     315732.51
es     309373.83
Name: total_cost, dtype: float64

In [129]:
master_df.groupby('customer_state')['delivery_time'] \
    .mean() \
    .sort_values()

customer_state
sp     8.136229
mg    11.406715
pr    11.429042
df    12.352578
rj    13.721933
sc    14.205622
rs    14.378461
go    14.497927
es    14.685476
ms    14.767089
pe    16.869958
to    16.980066
mt    17.158266
rn    17.534553
pi    17.609658
ba    17.944271
ce    18.908683
se    19.242340
ac    19.273810
pb    19.324864
ro    19.343396
ma    20.205298
rr    20.894737
pa    21.379381
al    22.466334
ap    23.721519
am    24.866242
Name: delivery_time, dtype: float64

In [130]:
master_df['order_estimated_delivery_date'] = pd.to_datetime(master_df['order_estimated_delivery_date'])

master_df['order_purchase_timestamp'] = pd.to_datetime(master_df['order_purchase_timestamp'])

In [131]:
master_df['estimated_delivery_time'] = (
    master_df['order_estimated_delivery_date'] - 
    master_df['order_purchase_timestamp']
).dt.days

In [132]:
late_delivery_pct = (
    master_df['delivery_time'] > master_df['estimated_delivery_time']
).mean()

late_delivery_pct

np.float64(0.06662584777694047)

In [133]:
(master_df['delivery_time'] > master_df['estimated_delivery_time']) \
.groupby(master_df['customer_state']) \
.mean() \
.sort_values(ascending=False)

customer_state
al    0.194514
ma    0.178808
se    0.139276
pi    0.128773
to    0.119601
ce    0.116018
ba    0.115724
es    0.111008
rj    0.105725
ms    0.094937
pb    0.088929
sc    0.084325
pa    0.077320
pe    0.075858
rn    0.075203
df    0.066108
go    0.063105
rs    0.056905
mt    0.052419
sp    0.050323
mg    0.047882
pr    0.042977
ro    0.041509
am    0.019108
ap    0.012658
ac    0.011905
rr    0.000000
dtype: float64

### Saved all Cleaned Files

In [134]:


# 1. Customers Clean
customers_clean = customers.copy()
customers_clean.drop_duplicates(inplace=True)

# Save
customers_clean.to_csv('customers_clean.csv', index=False)


# 2. Products Clean
products_clean = products.copy()

cols = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

for col in cols:
    products_clean[col] = products_clean[col].fillna(products_clean[col].median())

products_clean.to_csv('products_clean.csv', index=False)


# 3. Master Clean (MAIN DATASET)
master_clean = master_df.copy()

# Remove invalid values
master_clean = master_clean[master_clean['shipping_time'] >= 0]

# Cap outliers
master_clean['delivery_time'] = master_clean['delivery_time'].clip(upper=60)
master_clean['shipping_time'] = master_clean['shipping_time'].clip(upper=30)

# Create total cost
master_clean['total_cost'] = master_clean['price'] + master_clean['freight_value']

# Standardize column names
master_clean.columns = master_clean.columns.str.lower().str.replace(' ', '_')

# Save
master_clean.to_csv('master_clean.csv', index=False)


print("✅ All clean CSV files created successfully!")

✅ All clean CSV files created successfully!
